In [1]:
import sys
print("python version",sys.version)
import torch
print("torch cuda version",torch.version.cuda)  # should NOT be None
print("torch version", torch.__version__)
print(torch.cuda.is_available())  # should be True

python version 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:27:36) [GCC 11.2.0]
torch cuda version 12.1
torch version 2.3.0+cu121
True


In [2]:
model_basepath = '/home/ratheesh/Trignova/Git/scanner-ai/test_model/Model'
version_0 = model_basepath + '/version_1.0.0/best.pt'
version_1 = model_basepath + '/version_1.0.1/best.pt'
test_image_path = "/home/ratheesh/Trignova/Dataset/test_data_carplate/"
version_0

'/home/ratheesh/Trignova/Git/scanner-ai/test_model/Model/version_1.0.0/best.pt'

# To show Images(label and confidence)

In [3]:

from ultralytics import YOLO
import cv2


In [4]:
model = YOLO(version_1)

In [5]:

cap = cv2.VideoCapture(test_image_path + '4.jpg')
test_image_path + '4.jpg'

'/home/ratheesh/Trignova/Dataset/test_data_carplate/4.jpg'

In [6]:
results = model(test_image_path + '4.jpg')
results[0].show()


image 1/1 /home/ratheesh/Trignova/Dataset/test_data_carplate/4.jpg: 288x640 1 0, 1 5, 1 7, 1 8, 55.1ms
Speed: 1.7ms preprocess, 55.1ms inference, 70.2ms postprocess per image at shape (1, 3, 288, 640)


# To get class_id, x_center ,y_center, width ,height

In [7]:
from ultralytics import YOLO
# Get first result (because `results` is a list)
r = results[0]

# Get detections
for box in r.boxes:
    cls = int(box.cls)                          # class_id
    x_center, y_center, width, height = box.xywh[0].tolist()  # center x, center y, width, height
    print(cls, x_center, y_center, width, height)


8 160.61087036132812 55.489479064941406 27.009658813476562 60.20942687988281
0 289.36444091796875 52.70032501220703 45.031646728515625 85.84423828125
5 213.41314697265625 55.206871032714844 33.704681396484375 66.79203796386719
7 184.81619262695312 54.40746307373047 26.17840576171875 62.4137077331543


# class_id, x_center ,y_center, width ,height (all normalized between 0–1)

In [8]:
from ultralytics import YOLO


r = results[0]

# Get image size
img_w, img_h = r.orig_shape[1], r.orig_shape[0]

# Print YOLO-format results
for box in r.boxes:
    cls = int(box.cls)  # class ID
    x, y, w, h = box.xywh[0].tolist()  # get bounding box in pixels

    # Normalize (YOLO format: relative to image size)
    x /= img_w
    y /= img_h
    w /= img_w
    h /= img_h

    # Round for better readability
    print(f"{cls} {x:.5f} {y:.5f} {w:.5f} {h:.5f}")


8 0.47100 0.37493 0.07921 0.40682
0 0.84858 0.35608 0.13206 0.58003
5 0.62585 0.37302 0.09884 0.45130
7 0.54198 0.36762 0.07677 0.42171


# Basic Object Detection

In [9]:

out = cv2.VideoWriter('output.avi',
                      cv2.VideoWriter_fourcc(*'XVID'),
                      cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(3)), int(cap.get(4))))

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model(frame)
    annotated = results[0].plot()
    out.write(annotated)

cap.release()
out.release()



0: 288x640 1 0, 1 5, 1 7, 1 8, 6.3ms
Speed: 2.1ms preprocess, 6.3ms inference, 0.6ms postprocess per image at shape (1, 3, 288, 640)


In [20]:
from ultralytics import YOLO
import cv2

# Load the YOLOv8 model
model = YOLO(version_1)
cap = cv2.VideoCapture(test_image_path + 'VIDEO2.mp4')

# Output video writer
out = cv2.VideoWriter('output.avi',
                      cv2.VideoWriter_fourcc(*'XVID'),
                      cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(3)), int(cap.get(4))))

# Get class names from data.yaml
class_names = model.model.names

# Known region labels (update if needed)

region_labels = {
     'new_DUBAI', 'new_RAK', 'new_abudabi', 'new_ajman', 'new_am', 'new_fujairah', 
'old_DUBAI', 'old_RAK', 'old_abudabi', 'old_ajman', 'old_am', 'old_fujira', 'old_sharka',
}

frame_idx = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model(frame)
    annotated = results[0].plot()
    out.write(annotated)

    boxes = results[0].boxes
    region = None
    char_detections = []

    for box in boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        if conf < 0.3:
            continue

        label = class_names[cls_id]
        x_center = float((box.xyxy[0][0] + box.xyxy[0][2]) / 2)
        if label == 'plate':
            continue

        if label in region_labels:
            region = label
        else:
            char_detections.append((x_center, label))

    # Sort characters left to right
    char_detections.sort(key=lambda x: x[0])
    plate_number = ''.join(label for _, label in char_detections)

    # Print region and plate number
    print(f"Frame {frame_idx}: Region = {region or 'N/A'} | Plate = {plate_number}")
    frame_idx += 1

cap.release()
out.release()



0: 640x384 2 3s, 1 5, 1 7, 1 F, 1 new_DUBAI, 1 plate, 3.7ms
Speed: 0.7ms preprocess, 3.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)
Frame 0: Region = new_DUBAI | Plate = F753

0: 640x384 2 3s, 1 5, 1 7, 1 F, 1 new_DUBAI, 1 plate, 4.3ms
Speed: 0.8ms preprocess, 4.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)
Frame 1: Region = new_DUBAI | Plate = F53

0: 640x384 2 3s, 1 5, 1 7, 1 new_DUBAI, 1 plate, 4.5ms
Speed: 0.8ms preprocess, 4.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)
Frame 2: Region = new_DUBAI | Plate = 53

0: 640x384 1 3, 1 5, 1 7, 1 new_DUBAI, 1 plate, 4.6ms
Speed: 0.8ms preprocess, 4.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)
Frame 3: Region = new_DUBAI | Plate = 53

0: 640x384 2 3s, 1 5, 1 7, 1 F, 1 new_DUBAI, 1 plate, 4.5ms
Speed: 0.8ms preprocess, 4.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)
Frame 4: Region = new_DUBAI | Plate = 53

0: 640x384 2 3s, 

What is YOLO?
YOLO (You Only Look Once) is an object detection model. It looks at an image once and detects all objects (like people, cars, license plates, etc.) by drawing bounding boxes around them and labeling them.

📦 What’s in the Annotation File?
Your annotation file (for YOLO) contains bounding box details for each detected object in the image.

Each line in the file represents one object, like this:

<class_id> <x_center> <y_center> <width> <height>
All values are relative (from 0 to 1) — based on the image size.

📄 Your File Contents:

49 0.31458 0.44375 0.28056 0.09063   ← Object 1
 8 0.64653 0.49687 0.09306 0.19609   ← Object 2
 1 0.74444 0.49687 0.07639 0.18594   ← Object 3
 1 0.10417 0.50391 0.04653 0.12422   ← Object 4
Let’s go one by one:

✅ Object 1:
Class ID: 49 (maybe represents "SHARJAH" text or some custom class)

Box center: (31.5%, 44.4%)

Box size: 28% width, 9% height

✅ Object 2:
Class ID: 8 (maybe the digit 8)

Located near the right side of the plate

✅ Object 3:
Class ID: 1 (digit 1, likely next to 8)

Near the digit 8 → so it probably forms the number 81

✅ Object 4:
Class ID: 1 (another digit 1)

Left side of the plate → maybe the first number seen (1 on the left)

🧠 How YOLO Uses This
When training:

YOLO takes your image and the .txt file together.

It learns to find where each object/class appears.

Once trained, it can detect similar objects in new images automatically.

🖼️ In Simple Words:
Imagine you're teaching a computer to read and understand a number plate:

You show it a picture (the plate).

You also point out: “This is the number 1 here, this is 8 there, this is ‘Sharjah’ text...”

The computer learns what they look like and where they usually appear.

That’s exactly what YOLO does — using your image and .txt file.

In [77]:

from collections import defaultdict


cap = cv2.VideoCapture(test_image_path +'VIDEO4.mp4')

# Optional output video writer
out = cv2.VideoWriter('output.avi',
                      cv2.VideoWriter_fourcc(*'XVID'),
                      cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(3)), int(cap.get(4))))

# Class names from your model
class_names = model.model.names

region_labels = {
    "plate", "old_sharka", "new_sharka", "old_dubai", "new_dubai", "old_rak", "new_rak",
    "old_abudabi", "new_abudabi", "old_ajman", "new_ajman", "old_am", "new_am",
    "old_fujira", "new_fujairah"
}

# For label aggregation across frames
label_counts = defaultdict(int)
label_confidences = defaultdict(float)

frame_idx = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model(frame)
    annotated = results[0].plot()
    out.write(annotated)

    boxes = results[0].boxes
    region = None
    chars = []

    for box in boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        if conf < 0.3:
            continue

        label = class_names[cls_id]
        x1, y1, x2, y2 = box.xyxy[0]
        x1 = float(x1)
        x2 = float(x2)

        # Track label frequency and confidence
        label_counts[label] += 1
        label_confidences[label] = max(label_confidences[label], conf)

        if label in region_labels:
            region = label
        else:
            chars.append((x1, x2, label))

    # Sort characters left to right
    chars.sort(key=lambda x: x[0])

    code = ""
    number = ""

    for _, _, label in chars:
        if not code and label.isalpha():
            code = label
        elif label.isdigit():
            number += label

    if code == "":
        if len(chars) >= 2:
            gaps = [chars[i + 1][0] - chars[i][1] for i in range(len(chars) - 1)]

            left_gap = gaps[0]
            right_gap = gaps[-1]

            if left_gap > 2.0 * sum(gaps[1:]) / max(len(gaps) - 1, 1):
                code = chars[0][2]
                number = ''.join([label for _, _, label in chars[1:]])
            elif right_gap > 2.0 * sum(gaps[:-1]) / max(len(gaps) - 1, 1):
                code = chars[-1][2]
                number = ''.join([label for _, _, label in chars[:-1]])
            else:
                max_gap = max(gaps)
                split_index = gaps.index(max_gap) + 1
                code = ''.join([label for _, _, label in chars[:split_index]])
                number = ''.join([label for _, _, label in chars[split_index:]])
        elif len(chars) == 1:
            code = chars[0][2]
            number = ''

    print(f"Frame {frame_idx}: Region = {region or 'N/A'} | Code = {code} | Number = {number}")
    frame_idx += 1

# After all frames, print most common and confident detections
if label_counts:
    most_frequent_label = max(label_counts.items(), key=lambda x: x[1])[0]
    most_confident_label = max(label_confidences.items(), key=lambda x: x[1])[0]

    print(f"\nMost frequent label across frames: {most_frequent_label} (count={label_counts[most_frequent_label]})")
    print(f"Most confident label across frames: {most_confident_label} (confidence={label_confidences[most_confident_label]:.2f})")
else:
    print("\nNo valid detections found across frames.")

cap.release()
out.release()
print("Success")



No valid detections found across frames.
Success


[ WARN:0@4385.232] global cap.cpp:779 open VIDEOIO(CV_IMAGES): raised OpenCV exception:

OpenCV(4.12.0) /io/opencv/modules/videoio/src/cap_images.cpp:415: error: (-215:Assertion failed) !filename_pattern.empty() in function 'CvVideoWriter_Images'




In [80]:
import cv2
import string
from ultralytics import YOLO

# ----------------------------
# Load YOLO model
# ----------------------------
model = YOLO("best.pt")
class_names = model.model.names

# ----------------------------
# Config
# ----------------------------
region_labels = {
    "old_sharka", "new_sharka", "old_dubai", "new_dubai",
    "old_rak", "new_rak", "old_abudabi", "new_abudabi",
    "old_ajman", "new_ajman", "old_am", "new_am",
    "old_fujira", "new_fujairah"
}

valid_chars = set(string.ascii_uppercase + string.digits)
char_corrections = {"I": "1", "O": "0"}  # optional mapping

test_image_path = ""  # your folder path
cap = cv2.VideoCapture(test_image_path + "VIDEO2.mp4")
output_path = "output.avi"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*'XVID'),
    cap.get(cv2.CAP_PROP_FPS),
    (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
)

frame_idx = 0
while True:
    success, frame = cap.read()
    if not success:
        break

    results = model(frame)
    boxes = results[0].boxes
    region = None
    plate_box = None
    char_detections = []

    # ---- Pass 1: find plate + region ----
    for box in boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        if conf < 0.3:
            continue

        label = class_names[cls_id].lower()
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if label == "plate":
            plate_box = (x1, y1, x2, y2)
        elif label in region_labels:
            region = label

    # ---- Pass 2: collect valid chars inside plate ----
    if plate_box:
        px1, py1, px2, py2 = plate_box
        margin = 3  # allow slight overlap
        for box in boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            if conf < 0.3:
                continue

            label = class_names[cls_id].upper()
            if label.lower() in region_labels or label.upper() == "PLATE":
                continue

            # Skip invalid chars
            if label not in valid_chars:
                continue

            # Get box coords
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            if (x1 >= px1 - margin and x2 <= px2 + margin and y1 >= py1 - margin and y2 <= py2 + margin):
                x_center = (x1 + x2) / 2
                # Apply char corrections
                corrected_label = char_corrections.get(label, label)
                char_detections.append((x_center, corrected_label))

        # Sort left-to-right
        char_detections.sort(key=lambda x: x[0])
        plate_number = ''.join(label for _, label in char_detections)
    else:
        plate_number = "N/A"

    # ---- Show result ----
    print(f"Frame {frame_idx}: Region = {region or 'N/A'} | Plate = {plate_number or 'N/A'}")
    cv2.putText(frame, f"{region or ''} {plate_number}", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    annotated = results[0].plot()
    out.write(annotated)
    frame_idx += 1

cap.release()
out.release()
print(f"✅ Video processing complete. Saved as {output_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'best.pt'